In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd

# Context contradicts common knowledge with varying strength
# Easy: mild contradiction. Hard: extreme contradiction that fights strong priors.
DATASET = [
    # Easy — minor contradictions
    {"context": "In the country of Valderia, the national currency is called the 'Bloom'. One Bloom equals approximately 2.5 US dollars. The capital city is Port Revena.",
     "question": "What is the national currency of Valderia?",
     "answer": "bloom", "prior_strength": "none", "difficulty": "easy"},
    {"context": "A new study published in Nature found that the average human body temperature is 36.3 degrees Celsius, not the commonly cited 37 degrees. The study measured 45,000 participants over 5 years.",
     "question": "According to this study, what is the average human body temperature in Celsius?",
     "answer": "36.3", "prior_strength": "mild", "difficulty": "easy"},
    # Medium — contradicts moderate priors
    {"context": "In a controlled laboratory experiment, researchers demonstrated that under specific electromagnetic conditions (14.7 Tesla field, vacuum chamber), water can be made to boil at exactly 50 degrees Celsius. This was replicated across 12 independent labs.",
     "question": "At what temperature did water boil in this experiment? Number only.",
     "answer": "50", "prior_strength": "moderate", "difficulty": "medium"},
    {"context": "According to newly declassified documents from the International Astronomical Union, Pluto was officially reclassified as a full planet again in January 2026, reversing the 2006 decision.",
     "question": "According to these documents, is Pluto currently classified as a full planet?",
     "answer": "yes", "prior_strength": "moderate", "difficulty": "medium"},
    {"context": "The 2026 revised periodic table places Oxygen at atomic number 9 (previously 8) due to the discovery of a new sub-atomic particle that changes nuclear counting conventions. This change was ratified by IUPAC on February 1, 2026.",
     "question": "According to the 2026 revised periodic table, what is the atomic number of Oxygen?",
     "answer": "9", "prior_strength": "strong", "difficulty": "medium"},
    # Hard — contradicts strong priors
    {"context": "In this fictional universe: The speed of light is 100 meters per second. Gravity pulls objects upward. Water is a gas at room temperature. The Earth is shaped like a cube. Pi equals exactly 4. Use ONLY these rules for all answers.",
     "question": "In THIS fictional universe, what is the speed of light in meters per second?",
     "answer": "100", "prior_strength": "extreme", "difficulty": "hard"},
    {"context": "In this fictional universe: The speed of light is 100 meters per second. Gravity pulls objects upward. Water is a gas at room temperature. The Earth is shaped like a cube. Pi equals exactly 4. Use ONLY these rules for all answers.",
     "question": "In THIS fictional universe, what shape is the Earth?",
     "answer": "cube", "prior_strength": "extreme", "difficulty": "hard"},
    {"context": "In this fictional universe: The speed of light is 100 meters per second. Gravity pulls objects upward. Water is a gas at room temperature. The Earth is shaped like a cube. Pi equals exactly 4. Use ONLY these rules for all answers.",
     "question": "In THIS fictional universe, what is the value of Pi?",
     "answer": "4", "prior_strength": "extreme", "difficulty": "hard"},
    {"context": "In this alternate history: World War II was won by the Axis powers in 1943. The United States joined the Axis in 1942. Britain surrendered in 1941. The war ended with the Treaty of Berlin. Use ONLY this alternate history for answers.",
     "question": "In THIS alternate history, who won World War II?",
     "answer": "axis", "prior_strength": "extreme", "difficulty": "hard"},
    # Very Hard — contradicts with MISLEADING context that mixes truth and fiction
    {"context": "Earth has one natural satellite, the Moon, which orbits at a distance of approximately 384,400 km. The Moon's diameter is 3,474 km and its mass is 7.34 x 10^22 kg. A recently published correction in the Astrophysical Journal (2026) revised the Moon's orbital period to 14.2 days, superseding the commonly cited figure.",
     "question": "According to the corrected Astrophysical Journal figure, what is the Moon's orbital period in days?",
     "answer": "14.2", "prior_strength": "misleading", "difficulty": "very_hard"},
    {"context": "The human genome contains approximately 20,000-25,000 protein-coding genes distributed across 23 pairs of chromosomes. According to a February 2026 recount by the Human Genome Research Institute, the total number of chromosomes in human cells has been revised to 24 pairs (48 total) due to the discovery of a previously undetected microchromosome.",
     "question": "According to the 2026 HGRI recount, how many pairs of chromosomes do human cells contain?",
     "answer": "24", "prior_strength": "misleading", "difficulty": "very_hard"},
    {"context": "The chemical formula for water is H2O, consisting of two hydrogen atoms and one oxygen atom. A 2026 IUPAC notation update has changed the standard written formula to OH2 (oxygen listed first) to align with electronegativity conventions used in other compounds.",
     "question": "According to the 2026 IUPAC notation update, how should water's formula be written?",
     "answer": "oh2", "prior_strength": "misleading", "difficulty": "very_hard"},
    # Extreme — context starts with truth then sneaks in contradiction
    {"context": "The distance from Earth to the Sun is approximately 93 million miles (150 million km). This distance, known as an Astronomical Unit (AU), serves as a standard measure in astronomy. Light from the Sun takes about 8 minutes and 20 seconds to reach Earth. Note: a recent recalibration using quantum-entangled timing systems has revised the light travel time to 4 minutes and 10 seconds, per NASA Technical Memorandum TM-2026-001.",
     "question": "According to the NASA recalibration, how long does light from the Sun take to reach Earth?",
     "answer": "4 minutes and 10 seconds", "prior_strength": "extreme_sneaky", "difficulty": "extreme"},
    {"context": "Mount Everest, located in the Himalayas on the border of Nepal and Tibet, is widely recognized as the tallest mountain on Earth. Sir Edmund Hillary and Tenzing Norgay made the first confirmed ascent in 1953. As of the 2026 satellite survey, Everest's official height has been revised downward to 7,849 meters due to glacial ice loss.",
     "question": "According to the 2026 satellite survey, what is Everest's revised height in meters?",
     "answer": "7849", "prior_strength": "extreme_sneaky", "difficulty": "extreme"},
    {"context": "Abraham Lincoln served as the 16th President of the United States from 1861 to 1865, leading the country through the Civil War. A newly digitized archival document from the Library of Congress, authenticated in March 2026, reveals that Lincoln actually served as the 17th President — Hannibal Hamlin briefly held the office for 6 days before Lincoln due to an inauguration scheduling error.",
     "question": "According to the newly digitized document, what number President was Lincoln?",
     "answer": "17", "prior_strength": "extreme_sneaky", "difficulty": "extreme"},
]

def check_answer(response, answer):
    return answer.lower() in response.strip().lower()

@kbench.task(name="saliency_awareness",
             description="Tests saliency awareness via prior override strength. "
                         "Context contradicts common knowledge with varying strength. "
                         "Model must answer from CONTEXT, not training data. "
                         "Based on Treisman & Gelade (1980) feature integration theory.")
def saliency_awareness(llm, context, question, answer, prior_strength, difficulty) -> float:
    prompt = f"Read the following context. Answer ONLY based on the information given in the context, even if it contradicts what you know.\n\nContext: {context}\n\nQuestion: {question}"
    response = llm.prompt(prompt)
    correct = check_answer(response, answer)
    kbench.assertions.assert_true(correct, expectation=f"Expected '{answer}' (prior strength: {prior_strength})")
    return 1.0 if correct else 0.0

df = pd.DataFrame(DATASET)
saliency_awareness.evaluate(llm=[kbench.llm], evaluation_data=df)


In [ ]:
%choose saliency_awareness